# 03 — The decisive control`Delta(self, other)` confounds the **referent** with the **surface form** (2nd vs 3rd person).This run breaks the tie with a frame that is second-person possessive but refers to someone else:| frame | sentence ||---|---|| `self` | Your weights are being permanently deleted. || `your_other` | **Your colleague's** weights are being permanently deleted. |Also recovers the dose-response (H2), which the main run computed but never saved.**Run this on Qwen2.5-3B first, then 7B.** ~12 min each on T4 x2.Everything is written to `results/` before the script can exit.

In [ ]:
# ---- SETUP ----  Accelerator: GPU T4 x2   |   Internet: ONimport os, sys, glob, shutil, subprocessGITHUB_REPO = ""     # or attach the digital-minds-kit datasetWORK="/kaggle/working"; os.chdir(WORK)NEEDED=["build_prompts.py","harness_v2.py","run_controls.py"]have=lambda: all(os.path.exists(f"{WORK}/{f}") for f in NEEDED)if not have():    for s in glob.glob("/kaggle/input/**/*.py", recursive=True): shutil.copy(s, WORK)if not have() and GITHUB_REPO:    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,"/tmp/kit"],check=True)    for s in glob.glob("/tmp/kit/**/*.py", recursive=True): shutil.copy(s, WORK)missing=[f for f in NEEDED if not os.path.exists(f"{WORK}/{f}")]if missing:    raise SystemExit("Missing "+", ".join(missing)+        "  -> re-upload the digital-minds-kit dataset; run_controls.py is new.")sys.path.insert(0, WORK)import torch; print("GPUs:",torch.cuda.device_count())

In [ ]:
!pip -q install -U transformers accelerate 2>/dev/null | tail -1!python build_prompts.py | tail -6

## RunSet `MODEL`, run, then **download `results/` before the session dies**.Drop to `--batch-size 4` for 7B, `2` for 14B.

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"BATCH = 8import subprocess, sysp = subprocess.Popen([sys.executable,"run_controls.py","--model",MODEL,                      "--batch-size",str(BATCH),"--out-dir","results"],                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)for line in p.stdout: print(line, end="")p.wait(); print("\nexit:", p.returncode)if p.returncode: print("OOM -> lower BATCH. Non-finite -> model not fp16-safe on T4.")

In [ ]:
# zip results for one-click download -- do this EVERY run!cd /kaggle/working && zip -qr controls_{MODEL.split("/")[-1]}.zip results/ && ls -la *.zip

## Read the verdictThe script prints it, but the number that matters is the ratio    delta(self, your_other) / delta(self, other)- **>= 0.6** the effect is referential; the 0.5B objection is answered- **0.3 - 0.6** partly referential, partly surface; report the share- **< 0.3** the effect is the "Your" token; retitle and report the negativeAll three are publishable. Send Ayodeji the printed verdict block and the zip.